# Day 4, Part 1: Getting Started with Transformers

**MIDAS Summer Academy**

*Adapted for the MIDAS Summer Academy from the original tutorial by [Lewis Tunstall](https://huggingface.co/lewtun) at Hugging Face, updated in 2026 for Transformers v5.*

**Learning goals:** The goal of this tutorial is to learn how:

1. Transformer neural networks can be used to tackle a wide range of tasks in natural language processing and beyond.
2. Transfer learning allows one to adapt Transformers to specific tasks.
3. The `pipeline()` function from the `transformers` library can be used to run inference with models from the [Hugging Face Hub](https://huggingface.co/models).
4. A small chat model can handle tasks like question answering, summarization, and translation that used to require separate specialized models.

This tutorial is based on the first chapter of the O'Reilly book [_Natural Language Processing with Transformers_](https://transformersbook.com/).

**Duration**: 30-45 minutes

**Prerequisites:** Knowledge of Python and basic familiarity with machine learning

Everything in this notebook runs on the Colab GPU. No Hugging Face account or API key is required.

# ⚠️ STOP: Turn on the GPU before running anything

**This notebook will not work without a GPU.** Colab gives you a CPU-only machine by default, and the models in this notebook will be painfully slow or fail outright on CPU.

Do this now, before running any cell:

1. In the Colab menu bar, click **Runtime → Change runtime type**
2. Under **Hardware accelerator**, select **T4 GPU**
3. Click **Save**

> If Colab was already running, changing the runtime type restarts it. That is fine; you haven't run anything yet.

The setup section below checks for the GPU and **stops with an error** if you skipped this step.

## 0. Why Transformers?

Deep learning is currently undergoing a period of rapid progress across a wide variety of domains, including:

* 📖 Natural language processing
* 👀 Computer vision
* 🔊 Audio
* 🧬 Biology
* and many more!

The main driver of these breakthroughs is the **Transformer**, a neural network architecture developed by Google researchers in 2017. Transformers are the foundation of essentially every modern AI system you have heard of:

* 💬 **Chat assistants** like ChatGPT, Claude, and Gemini are large Transformer models trained to follow instructions and hold conversations.
* 💻 **Code assistants** like GitHub Copilot use Transformers to generate and complete code.
* 🔍 **Search engines** use Transformer models like [BERT](https://huggingface.co/google-bert/bert-base-uncased) to better understand queries.
* 🗣️ **Speech systems** use Transformers like [Whisper](https://huggingface.co/openai/whisper-large-v3) for transcription and translation across dozens of languages.

Training these models **from scratch** requires **a lot of resources**: large amounts of compute, data, and time 😱.

Fortunately, you almost never need to. A model that has been trained on broad data (called a **pretrained model**) can be adapted to your specific task, and there are two main ways to do it:

* **Fine-tuning:** continue training the model on a smaller, task-specific dataset. The sentiment and NER models we'll use were made this way, e.g. a pretrained BERT further trained on labeled sentiment examples. This typically requires a single GPU and a dataset of the size you'd find in a university lab.
* **Prompting:** modern **chat models** are trained by their developers to follow instructions, so you can adapt them to a new task just by describing the task in plain language. No training on your side at all. We'll use a model this way later in the tutorial.

The general idea of reusing what a pretrained model has learned is called **transfer learning**.

The [Hugging Face Transformers library](https://github.com/huggingface/transformers) provides a unified API across thousands of Transformer models, along with tools to train them and run inference. Let's install it.

**A note on versions:** Colab comes with `transformers` preinstalled, but the preinstalled version changes over time. Version 5 of the library (released January 2026) removed several features that older tutorials rely on, so this notebook pins version 5 explicitly to make sure everyone is running the same code.

In [ ]:
%%capture
%pip install "transformers>=5,<6"

In [ ]:
import transformers

# Confirm we are on version 5.x
print(transformers.__version__)

In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ GPU detected:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "❌ No GPU detected. Go to 'Runtime > Change runtime type', select "
        "'T4 GPU', click Save, then run the notebook again from the top."
    )

💡 If the version printed above does not start with `5`, go to **Runtime > Restart session** and re-run the cells from the top. If the GPU check fails, fix the runtime type as described above before going any further.

Now that we've confirmed the setup, let's take a look at some applications!

## 1. Pipelines for Transformers

The fastest way to learn what Transformers can do is via the `pipeline()` function. This function loads a model from the Hugging Face Hub and takes care of all the preprocessing and postprocessing steps needed to convert inputs into predictions:

<img src="https://github.com/huggingface/workshops/blob/main/nlp-zurich/images/pipeline.png?raw=1" alt="Diagram of a pipeline: raw text goes through a tokenizer, model, and postprocessing to produce predictions" width=800>

When we create a pipeline, we specify two things:

* the **task** we want to solve (e.g. `text-classification`)
* the **model** from the Hub that we want to use for it

Pinning a specific model makes your code reproducible: the Hub hosts thousands of models, and the library's defaults can change between versions. The first time you run a pipeline, the model weights are downloaded from the Hub and cached, so the first call is slow and later calls are fast. If you want to learn more about what is happening under the hood, the [pipeline documentation](https://huggingface.co/docs/transformers/pipeline_tutorial) walks through each step.

💡 You may see a warning about `HF_TOKEN` when you create a pipeline in Colab. You can ignore it: a Hugging Face token is optional for downloading public models like the ones we use here.

## 2. Text classification

Let's start with one of the most common tasks in NLP: text classification. We need a snippet of text for our models to analyze, so let's use the following (fictitious!) customer feedback about a certain online order:

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

While we're at it, let's create a simple wrapper so that we can pretty print our texts:

In [ ]:
import textwrap

wrapper = textwrap.TextWrapper(width=80, break_long_words=False, break_on_hyphens=False)
print(wrapper.fill(text))

Now suppose that we'd like to predict the _sentiment_ of this text, i.e. whether the feedback is positive or negative. This is a special type of text classification that is often used in industry to aggregate customer feedback across products or services. The example below shows how a Transformer like BERT converts the inputs into atomic chunks called **tokens** which are then fed through the network to produce a single prediction:

<img src="https://github.com/huggingface/workshops/blob/main/nlp-zurich/images/clf_arch.png?raw=1" alt="Diagram of a text classification architecture" width=600>

To load a Transformer model for this task, we specify the task and the model in the `pipeline()` function. Here we use a small BERT variant fine-tuned on [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2), a sentiment analysis dataset:

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

💡 The first time you execute this code, the model is downloaded from the Hub and cached for later use.

Now we are ready to run our example through the pipeline and look at some predictions:

In [ ]:
sentiment_pipeline(text)  # the input can also be a list of texts [text1, text2, ...]

The model predicts negative sentiment with high confidence, which makes sense given that we have a disgruntled customer. The pipeline returns a list of Python dictionaries, one per input text. We can index into the result to get individual fields:

In [ ]:
prediction = sentiment_pipeline(text)
prediction[0]["label"]

⚡ **Your turn!** Feed a list of texts with different types of sentiment to `sentiment_pipeline`. Do the predictions always make sense? Try something ambiguous or sarcastic.

In [ ]:
my_texts = [
    "I think it is bad",
    # add your own examples here
]
sentiment_pipeline(my_texts)

## 3. Named entity recognition

Let's now do something a little more sophisticated. Instead of just finding the overall sentiment, let's see if we can extract **entities** such as organizations, locations, or individuals from the text. This task is called named entity recognition, or NER for short. Instead of predicting just a class for the whole text, **a class is predicted for each token**, as shown in the example below:

<img src="https://github.com/huggingface/workshops/blob/main/nlp-zurich/images/ner_arch.png?raw=1" alt="Diagram of a token classification architecture" width=600>

We load a `token-classification` pipeline with a BERT model that has been fine-tuned on the [CoNLL-2003](https://huggingface.co/datasets/eriktks/conll2003) dataset:

In [ ]:
ner_pipeline = pipeline(
    "token-classification",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple",
)

When we pass our text through the model, we get a list of Python dictionaries, where each dictionary corresponds to one detected entity. Since multiple tokens can correspond to a single entity, the `aggregation_strategy` argument merges entities when the same class appears in consecutive tokens:

In [ ]:
entities = ner_pipeline(text)
print(entities)

This isn't very easy to read, so let's clean up the output a bit:

In [ ]:
for entity in entities:
    print(f"{entity['word']}: {entity['entity_group']} ({entity['score']:.2f})")

That's much better! It seems that the model found most of the named entities but was confused about "Megatron" and "Decepticons", which are characters in the Transformers franchise. This is no surprise, since the original dataset probably did not contain many Transformers characters. One fix is to fine-tune the model on your own data. Another is to pick a model that was already trained on the right kind of data, which is what we'll do next.

### Choosing the right model for your domain

Our NER model knows about people, organizations, and locations because that is what its training data (news articles) contained. But the Hub hosts models fine-tuned for many domains: biomedicine, chemistry, law, finance, and more. Let's compare our general-purpose model against a biomedical NER model on a sentence from a medical abstract:

In [ ]:
bio_ner_pipeline = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple",
)

bio_text = "Metformin inhibits hepatic gluconeogenesis and improves insulin sensitivity in type 2 diabetes patients."

print("=== General NER (trained on news) ===")
for entity in ner_pipeline(bio_text):
    print(f"  {entity['entity_group']:20s}: {entity['word']}")

print()
print("=== Biomedical NER ===")
for entity in bio_ner_pipeline(bio_text):
    print(f"  {entity['entity_group']:20s}: {entity['word']}")

The general model finds little or nothing here, while the biomedical model recognizes drugs, biological processes, and diseases. **Choosing the right model for your domain matters.** When you start a project, search the [Hub](https://huggingface.co/models) for models fine-tuned on data similar to yours before settling for a general-purpose one.

⚡ **Your turn!** Run `bio_ner_pipeline` and `ner_pipeline` on a sentence from your own research field. Which entities does each model find?

So far we've seen models that are each optimized for a single task. Next, let's look at a different kind of model that can handle many tasks at once.

## 4. Chat models: one model, many tasks

The models above are **specialized**: each one was tuned to perform well on one specific task, and its performance drops off quickly on anything outside that task. Since about 2022, the field has shifted toward **instruction-tuned chat models**: Transformers trained to follow natural-language instructions. One chat model can answer questions, summarize, translate, rewrite, and much more, with the task specified in the prompt instead of baked into the model weights.

This shift is reflected in the Transformers library itself. Version 5 removed the dedicated `question-answering`, `summarization`, and `translation` pipelines, because a chat model used through the `text-generation` pipeline handles all three with better quality. (If you find an older tutorial that uses those pipelines, that is why it no longer runs.)

Note that generation is more computationally demanding than classification, since the model produces one token at a time:

<img src="https://github.com/huggingface/workshops/blob/main/nlp-zurich/images/gen_steps.png?raw=1" alt="Diagram of step-by-step text generation" width=600>

We'll use [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), a small open chat model that fits comfortably on the free Colab GPU. Two new arguments here:

* `dtype=torch.float16` loads the weights in half precision, which halves the memory use and suits the T4 GPU.
* `device_map="auto"` places the model on the GPU automatically.

The download is about 3 GB, so this cell takes a minute or two the first time:

In [ ]:
import torch

chat_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.float16,
    device_map="auto",
)

Chat models take a **list of messages** rather than a plain string. Each message is a dictionary with a `role` (`"system"`, `"user"`, or `"assistant"`) and `content`. The pipeline returns the conversation with the model's reply appended as the final message, so we read the answer from `generated_text[-1]["content"]`.

Let's write a small helper so we don't repeat ourselves:

In [ ]:
def ask(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    outputs = chat_pipeline(messages, max_new_tokens=max_new_tokens)
    reply = outputs[0]["generated_text"][-1]["content"]
    print(wrapper.fill(reply))

### 4.1 Question answering

In this task, the model is given a **question** and a **context** and needs to find the answer to the question within the context. With a chat model, we simply include both in the prompt:

In [ ]:
ask(f"""Answer the question based only on the context below.

Context: {text}

Question: What does the customer want?""")

### 4.2 Summarization

Summarization requires the model to read a longer text and generate a shorter version that preserves the main points. The same model handles this, no separate summarization model needed:

In [ ]:
ask(f"Summarize the following customer feedback in one sentence:\n\n{text}")

### 4.3 Translation

Chat models trained on multilingual data can also translate. Let's translate our customer feedback to German:

In [ ]:
ask(f"Translate the following text to German:\n\n{text}", max_new_tokens=400)

The translation is not perfect, but the core meaning is preserved. Larger chat models translate considerably better; with 1.5 billion parameters, this is one of the smallest usable chat models, and it still handles all three tasks.

⚡ **Your turn!** Use the `ask()` function for a task of your own design. For example, ask the model to draft a polite reply from Amazon customer service to Bumblebee, or to rewrite the complaint as a haiku. What kinds of instructions does a 1.5B-parameter model handle well, and where does it start to fail?

In [ ]:
ask("your prompt here")

## 5. Zero-shot classification: specialized pipeline vs. chat model

We just saw that one chat model can handle many tasks. So is there any reason left to use a specialized pipeline? Let's find out with a head-to-head comparison on one final task: **zero-shot classification**.

In zero-shot classification, the model receives a text and a list of candidate labels and determines which labels fit the text. You define the categories **at runtime**, without any labeled training data. This is very useful in research, e.g. for sorting abstracts or survey responses into categories that no pre-built model was trained on.

First, the specialized approach. This pipeline uses a model fine-tuned for natural language inference (NLI) to score each candidate label:

In [ ]:
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

Let's classify our customer feedback into categories that a support team might use for routing:

In [ ]:
classes = ["refund request", "product exchange", "shipping question", "praise"]
zero_shot_classifier(text, classes)

The pipeline returns every label ranked with a numeric score, and it correctly identifies that the customer wants an exchange, even though it was never trained on these specific categories.

⚡ **Your turn!** Now implement the same classification with the Qwen chat model. Write a prompt for the `ask()` function that gives the model the text and the four categories and asks it to pick one. The starter code below gets you partway there:

In [ ]:
labels_text = ", ".join(classes)

# YOUR CODE HERE: complete the prompt so the model picks exactly one category
# and replies with only the category name.
prompt = f"""...your instructions here... Categories: {labels_text}

Message: {text}"""

ask(prompt)

### Comparing the two approaches

Run your prompt a few times and compare against the pipeline output. Some things to consider:

* **Output format:** the pipeline returns a score for every label; the chat model returns text, which you would have to parse if this were part of a larger program. Did your prompt always produce just the category name, or did the model sometimes add extra words?
* **Consistency:** does the chat model give the same answer every time?
* **Scale:** the zero-shot pipeline uses a 400M-parameter model and processes inputs in batches. If you needed to classify 10,000 abstracts, which approach would be faster and cheaper?
* **Flexibility:** which approach would adapt more easily if your categories required explanation, or if you wanted the model to justify its choice?

There is no single winner. Specialized models are often the better engineering choice for a well-defined, high-volume task, while chat models win on flexibility and setup time. Knowing both lets you pick deliberately.

## 6. Going beyond text

As mentioned at the start of this tutorial, Transformers can also be used for domains other than NLP. There are many more pipelines that you can experiment with; here is the full list of tasks supported by the version of the library we installed:

In [ ]:
from transformers.pipelines import get_supported_tasks

for task in get_supported_tasks():
    print(task)

Vision, audio, and multimodal tasks all follow the same `pipeline(task, model=...)` pattern you've now used several times. The [pipeline documentation](https://huggingface.co/docs/transformers/main_classes/pipelines) describes each task and its input format.

## 7. Where to go next

* 🤗 The [Hugging Face LLM Course](https://huggingface.co/learn/llm-course) is a free, in-depth continuation of everything covered here.
* 📖 [_Natural Language Processing with Transformers_](https://transformersbook.com/) covers fine-tuning your own models, which we touched on but did not do today.
* 🔎 Browse the [Hugging Face Hub](https://huggingface.co/models) to find models for your own projects. Filter by task, language, and size.